# Cohere Rerank를 이용한 2단계 검색

Reranking은 빠른 1차 검색기가 넓게 모은 후보를 더 정교한 모델로 다시 정렬하는 2단계 검색 전략이다. BM25와 Dense Retrieval은 후보를 모아 Recall을 확보하고, Cohere Rerank는 질의와 각 후보 본문을 함께 읽어 후보 내부의 상위 순서를 정교하게 만든다.

이 단원은 핵심 검색 경로를 익힌 뒤 필요에 따라 선택하는 독립 심화 분기이다. 실행에는 BM25·Dense 검색의 기본 개념, 원문 문서가 적재된 Pinecone 인덱스, OpenAI·Pinecone·Cohere 설정이 필요하며 패키지와 CSV는 이 노트북에서 준비한다. BM25·Dense 후보를 직접 만들므로 HyDE 결과를 입력으로 사용하지 않으며, 후보 내부 순서가 병목일 때 Rerank를 선택한다.

두 단계의 책임은 분리된다. 1차 검색이 정답 문서를 후보에 넣지 못하면 Rerank는 그 문서를 새로 찾거나 복구할 수 없다. 따라서 후보 집합의 Recall을 먼저 확보한 뒤, 제한된 후보 안에서 상위 정밀도와 순서를 개선해야 한다.

실제 매핑 흐름은 `query → BM25·Dense 후보 ID → candidates → 같은 순서의 documents → ClientV2.rerank() → response.results의 index → candidates[index] → doc_id`이다. 각 result의 `relevance_score`는 해당 질의·모델·후보 집합 안에서 순서를 정하는 점수이며, 정답 확률이 아니고 BM25나 Pinecone의 원점수와 직접 비교하지 않는다.


## Cohere Rerank 실행 패키지 준비

`%pip`은 현재 Jupyter 커널에 필요한 패키지를 설치한다. BM25·Dense 후보 생성, Cohere v2 SDK와 검색 평가 패키지를 준비한다. 설치 후 커널이 이전 모듈을 잡고 있으면 한 번 재시작한다.

### 코드 해석 순서

1. Cohere Rerank 실습에 필요한 공식 패키지를 현재 커널에 설치한다.

### 결과 해석

- 패키지별 설치 로그가 나타나며 의존성 충돌이 없으면 셀이 종료된다.
- 설치가 끝나면 다음 셀에서 Cohere Rerank의 모델·검색기 객체를 직접 import할 수 있다.


In [ ]:
# 설치 목록은 Cohere Rerank 활성 코드에 필요한 SDK와 분석 도구를 포함한다.
# `-U`는 이미 설치된 패키지를 호환되는 최신 배포본으로 갱신한다.
%pip install -U pandas numpy rank_bm25 konlpy langchain langchain-openai langchain-pinecone pinecone cohere python-dotenv gdown tqdm


## Cohere Rerank 환경 변수와 모델 이름 준비

`load_dotenv()`는 `.env`에 저장한 OpenAI·Pinecone·Cohere 설정을 현재 Python 환경으로 불러온다. API key는 출력하지 않으며 각 SDK가 환경 변수에서 사용한다.

이 노트북에서 직접 사용하는 설정만 준비한다.

- `PINECONE_INDEX_NAME`: 원문이 저장된 Pinecone index 이름이다.
- `OPENAI_EMBEDDING_MODEL`: Dense 후보 검색에 사용할 임베딩 모델이다.
- `COHERE_RERANK_MODEL`: 후보 문서의 순서를 다시 계산할 Cohere 모델이다.

### 코드 해석 순서

1. `.env`를 불러오고 Cohere Rerank에서 직접 사용할 모델과 index를 지정한다.

### 결과 해석

- 환경 변수와 실습용 모델 설정이 준비되며 화면에 별도 출력은 나타나지 않는다.
- 후보 검색에는 Pinecone·OpenAI 설정을, 후보 재정렬에는 Cohere 설정을 사용한다.


In [ ]:
import os
from dotenv import load_dotenv

# load_dotenv()는 `.env`의 값을 환경 변수에 추가하며 API key를 화면에 출력하지 않는다.
load_dotenv()

# index 이름과 임베딩 모델은 Dense 후보를 검색할 때 사용한다.
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "adv-rag")
OPENAI_EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"
)

# Rerank 모델 이름은 뒤의 ClientV2.rerank()에 전달한다.
COHERE_RERANK_MODEL = os.getenv("COHERE_RERANK_MODEL", "rerank-v4.0-pro")


## 재정렬용 문서·질의 다운로드

문서 corpus와 30개 질의·정답 파일을 저장한다. 1차 검색과 Rerank는 같은 문서 ID를 유지해야 `response.results[index]`를 원래 문서로 복원할 수 있다.

### 코드 해석 순서

1. 후보 생성과 재정렬 평가에 사용할 두 CSV를 내려받는다.

### 결과 해석

- 실행하면 두 파일의 Google Drive 다운로드 완료가 나타난다.
- 문서 ID와 본문의 대응이 변하지 않아야 Cohere 응답 인덱스를 올바른 ID로 되돌릴 수 있다.


In [ ]:
# 문서 파일은 ID에서 Cohere에 보낼 본문 문자열을 찾는 조회표이다.
# 질의 파일은 후보 생성 입력과 최종 순위 지표의 정답을 제공한다.
# documents.csv
!gdown 1pspaw_q4_QCp2K4M-thDlUtra-_wQI7k
# queries.csv
!gdown 1dsn0pwkfzOUxiQ4MKDIvMM-CkxbYiSce


## 후보 문서와 질의 불러오기

두 CSV를 DataFrame으로 읽고 질의 표를 표시한다. `documents_df`는 후보 ID를 Cohere에 보낼 본문으로 바꾸는 조회표이고, `queries_df`는 질의 반복과 qrels(query relevance judgments) 평가에 사용된다.

`doc_id`와 후보 목록의 위치가 끝까지 유지되어야 Cohere 응답의 `index`를 올바른 문서 ID로 복원할 수 있다.

### 코드 해석 순서

1. documents_df는 후보 doc_id를 Cohere 입력 본문으로 바꾸는 조회표이다.
2. queries_df의 query_text는 두 검색기와 Rerank에, qrels는 평가에 사용된다.

### 결과 해석

- 실행하면 Q1부터 Q30까지 질의와 관련 문서 등급 열이 표시된다.
- 같은 질문과 정답을 사용하면 순위 변화가 Rerank 단계에서 비롯됐는지 판단할 수 있다.


In [ ]:
import pandas as pd

# 1. documents_df는 후보 doc_id를 Cohere 입력 본문으로 바꾸는 조회표이다.
documents_df = pd.read_csv('documents.csv')

# 2. queries_df의 query_text는 두 검색기와 Rerank에, qrels는 평가에 사용된다.
queries_df = pd.read_csv('queries.csv')
queries_df


## Rerank 후보용 BM25 검색기

BM25는 질문과 문서의 형태소가 얼마나 잘 겹치는지 계산하는 희소 검색기이다. Okt로 문서와 질문을 같은 방식으로 토큰화하고 점수가 높은 행의 `doc_id`를 반환한다.

`bm25_search()`의 출력은 `list[str]` 형태의 후보 ID 목록이다. 최종 5개보다 넓은 20개를 뽑아 Rerank가 순서를 바꿀 여지를 주지만, 이 후보 밖 문서는 뒤 단계에서 복구할 수 없다.

### 코드 해석 순서

1. 문서 본문을 형태소 목록으로 바꿔 BM25 corpus를 만든다.
2. 질문을 같은 방식으로 토큰화하고 점수 상위 문서 ID를 list[str]로 반환한다.

### 결과 해석

- BM25 객체와 함수가 생성되며 후보 결과는 뒤의 반복 셀에서 채운다.
- Rerank는 후보 밖 문서를 복구하지 못하므로 첫 단계의 Recall이 중요하다.


In [ ]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

# 1. 문서 본문을 형태소 목록으로 바꿔 BM25 corpus를 만든다.
okt = Okt()
tokenized_docs = [okt.morphs(content) for content in documents_df['content']]
bm25 = BM25Okapi(tokenized_docs)

# 2. 질문을 같은 방식으로 토큰화하고 점수 상위 문서 ID를 list[str]로 반환한다.
def bm25_search(query, top_k=5):
    query_token = okt.morphs(query)
    scores = bm25.get_scores(query_token)
    sorted_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    # 점수 인덱스를 원래 DataFrame의 doc_id로 복원해 후보 딕셔너리에 저장한다.
    ranked_docs = [documents_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs


## Rerank 후보용 Dense 검색기

`OpenAIEmbeddings`는 질문을 문서와 같은 벡터 공간으로 바꾸고, `PineconeVectorStore`는 지정한 index에서 의미적으로 가까운 `list[Document]`를 검색한다.

`index_name`은 검색할 Pinecone index, `embedding`은 질문 벡터 변환기이다. 이 코드에는 `namespace` 인자가 없으므로 기본 namespace를 사용한다. 각 `Document.metadata['doc_id']`를 꺼내 BM25와 같은 후보 ID 목록으로 맞춘다.

### 코드 해석 순서

1. model은 질문을 문서 색인과 같은 벡터 공간으로 바꿀 임베딩 모델이다.
2. index_name은 검색할 Pinecone index이고 embedding은 질문 벡터 변환기이다.

### 결과 해석

- 클라이언트 객체가 생성되고 이 셀 자체에는 검색 출력이 없다.
- similarity_search()의 list[Document]를 doc_id 목록으로 바꾸면 BM25 후보와 결합할 수 있다.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. model은 질문을 문서 색인과 같은 벡터 공간으로 바꿀 임베딩 모델이다.
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 2. index_name은 검색할 Pinecone index이고 embedding은 질문 벡터 변환기이다.
# namespace 인자를 전달하지 않으므로 similarity_search()는 기본 namespace를 사용한다.
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)


## 질의별 BM25·Dense 후보 20개 생성

각 질의에 두 1차 검색기를 독립 적용해 `bm25_candidates`와 `dense_candidates`를 만든다. 두 딕셔너리의 value는 높은 순위부터 정렬된 문서 ID 20개이다.

이 단계가 후보 집합의 Recall 상한을 정한다. Rerank는 다음 셀에서 두 목록을 합친 후보 안에서만 순서를 바꿀 수 있다.

### 코드 해석 순서

1. bm25_candidates[qid]에는 키워드 순위의 doc_id 20개를 저장한다.
2. Dense의 list[Document]를 metadata의 doc_id 20개로 변환한다.

### 결과 해석

- Pinecone 요청이 끝나면 두 후보 딕셔너리가 채워지며 이 셀은 별도 표를 출력하지 않는다.
- 두 목록의 합집합 Recall이 낮으면 어떤 재정렬 모델도 정답을 상위로 올릴 수 없다.


In [ ]:
# 1. bm25_candidates[qid]에는 키워드 순위의 doc_id 20개를 저장한다.
bm25_candidates = {}
for idx, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    bm25_candidates[qid] = bm25_search(query_text, top_k=20)

# 2. Dense의 list[Document]를 metadata의 doc_id 20개로 변환한다.
dense_candidates = {}
for idx, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    docs = vector_store.similarity_search(query_text, k=20)
    dense_candidates[qid] = [doc.metadata['doc_id'] for doc in docs]


## Cohere ClientV2 재정렬 함수

`ClientV2.rerank()`의 `model`은 재정렬 모델, `query`는 원문 질문, `documents`는 후보 본문 문자열 목록, `top_n`은 반환할 result 수이다. 여기서는 `top_n=len(texts)`로 모든 후보의 재정렬 결과를 받고, 호출부에서 상위 5개만 선택한다.

`texts[i]`와 `candidates[i]`는 같은 문서를 가리킨다. 따라서 `response.results`의 각 `result.index`를 `candidates[result.index]`에 적용하면 Cohere 입력 위치를 원래 `doc_id`로 복원할 수 있다. `result.relevance_score`는 같은 호출 안의 순위 해석에 쓰며 BM25·Pinecone 원점수와 직접 비교하지 않는다.

HTTP 429는 짧은 시간에 허용된 요청량을 넘겼다는 뜻이다. 함수는 지정한 시간만큼 기다렸다가 같은 요청을 한 번 더 시도하고, 마지막 실패는 숨기지 않고 호출자에게 전달한다.

### 코드 해석 순서

1. texts[i]와 candidates[i]가 같은 문서를 가리키도록 후보 ID를 본문으로 변환한다.
2. 최초 호출과 최대 max_retries번의 재시도를 수행한다.

### 결과 해석

- Cohere Client와 재정렬 함수가 정의되며 이 셀에서는 API 요청이 발생하지 않는다.
- 실제 요청은 뒤의 반복문이 함수를 호출할 때 발생하고 result.index로 문서 ID를 복원한다.


In [ ]:
import time

import cohere
from cohere import TooManyRequestsError

# ClientV2는 `.env`에서 불러온 COHERE_API_KEY로 Rerank endpoint에 연결한다.
co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])


def cohere_rerank(query, candidates, max_retries=1, wait_seconds=10):
    '후보 ID를 Cohere 관련성 순서의 문서 ID 목록으로 변환한다.'
    # 1. texts[i]와 candidates[i]가 같은 문서를 가리키도록 후보 ID를 본문으로 변환한다.
    texts = [
        documents_df.loc[documents_df["doc_id"] == doc_id, "content"].iloc[0]
        for doc_id in candidates
    ]

    # 2. 최초 호출과 최대 max_retries번의 재시도를 수행한다.
    for attempt in range(max_retries + 1):
        try:
            response = co.rerank(
                # model은 후보 관련성을 계산할 Cohere Rerank 모델이다.
                model=COHERE_RERANK_MODEL,
                # query는 모든 후보와 비교할 원문 질문이다.
                query=query,
                # documents는 candidates와 같은 위치를 유지하는 본문 문자열 목록이다.
                documents=texts,
                # top_n은 모든 후보 result를 받아 호출부가 상위 5개를 고르게 한다.
                top_n=len(texts),
            )
            # result.index를 candidates의 같은 위치에 적용해 원래 doc_id로 복원한다.
            return [candidates[result.index] for result in response.results]
        except TooManyRequestsError:
            # 마지막 재시도도 실패하면 오류를 숨기지 않고 호출자에게 전달한다.
            if attempt >= max_retries:
                raise
            print(
                f"TooManyRequestsError: {wait_seconds}초 후 재시도한다. "
                f"(시도 {attempt + 1}/{max_retries})"
            )
            time.sleep(wait_seconds)


## 후보 통합과 질의별 Rerank

BM25·Dense 목록을 이어 붙인 뒤 `dict.fromkeys()`로 첫 등장 순서를 유지하며 중복을 제거한다. 최대 40개의 후보 ID가 `candidates`가 되고, 같은 위치의 본문 목록이 Cohere 입력이 된다.

Cohere가 반환한 전체 재정렬 중 상위 5개 ID만 `rerank_results`에 저장한다. 질의 사이의 6초 대기는 요청 속도를 낮추는 사전 간격이고, 함수 안의 10초 대기는 실제 429 응답 뒤 재시도하는 간격이다. 어느 방식도 요금제 한도를 보장하지는 않는다.

### 코드 해석 순서

1. 두 순위를 이어 붙이고 첫 등장 순서를 유지하며 중복 ID를 제거한다.
2. response index로 복원한 ID 중 상위 5개를 RAG context 후보와 평가 입력으로 보존한다.

### 결과 해석

- 유료 경로를 실행하면 진행률이 30건까지 증가하고 질의별 Rerank ID 목록이 채워진다.
- 재정렬 품질뿐 아니라 질의당 후보 수, 대기 시간과 API 호출 비용이 운영 성능을 결정한다.


In [ ]:
from tqdm.auto import tqdm

rerank_results = {}

for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row["query_id"]
    query_text = row["query_text"]

    # 1. 두 순위를 이어 붙이고 첫 등장 순서를 유지하며 중복 ID를 제거한다.
    concat_candidates = bm25_candidates[qid] + dense_candidates[qid]
    candidates = list(dict.fromkeys(concat_candidates))

    # 2. response index로 복원한 ID 중 상위 5개를 RAG context 후보와 평가 입력으로 보존한다.
    rerank_results[qid] = cohere_rerank(query_text, candidates)[:5]
    # 질의 사이의 6초 간격은 요청 속도를 낮추지만 429 방지를 보장하지는 않는다.
    time.sleep(6)  # 수업 계정의 호출 한도를 완화하며 실제 요금제에 맞게 조정한다.


## Rerank 문서 ID 결과 확인

`rerank_results`는 `query_id → 관련성 순서의 doc_id 5개` 딕셔너리이다. 이 형식은 BM25·Dense 결과와 같으므로 다음 평가 함수에 바로 전달할 수 있고, 이후 RAG에서는 ID에 해당하는 실제 문서 본문을 context 후보로 조립할 수 있다.

### 코드 해석 순서

1. 모든 질의의 Cohere 재정렬 결과 구조를 확인한다.

### 결과 해석

- 외부 Rerank 경로를 실행하면 질의 ID별로 관련성 순서의 문서 ID 다섯 개가 표시된다.
- 각 ID는 1차 후보 안에서 선택되며 다음 평가와 실제 문서 context 조립에 사용할 수 있다.


In [ ]:
# 딕셔너리 key는 질의 ID이고 value 순서는 높은 관련성부터 낮은 관련성이다.
# Q1 표본에서 후보 집합의 정답 문서가 실제로 앞쪽으로 이동했는지 확인한다.
rerank_results


## Rerank 순위 평가 함수 정의

세 검색 결과를 같은 qrels와 같은 `k=5` 기준으로 평가한다. P@k는 상위 k 중 관련 문서 비율, R@k는 전체 관련 문서 중 상위 k가 찾은 비율, MRR은 첫 관련 문서 순위의 역수, MAP는 관련 문서가 나타날 때마다 계산한 정밀도의 평균이다.

함수 입력은 `query_id → list[doc_id]` 딕셔너리와 질의 DataFrame이다. AP@k의 분모에 상위 k에서 놓친 관련 문서도 반영해 재정렬 개선을 과대평가하지 않는다.

### 코드 해석 순서

1. qrels 문자열을 관련 문서 ID와 등급의 딕셔너리로 변환한다.
2. 예측 ID 순서와 정답 딕셔너리에서 P@k, R@k, RR와 AP@k를 계산한다.
3. 모든 질의의 네 지표를 평균내 검색 방식별 비교 딕셔너리를 반환한다.

### 결과 해석

- 함수 정의만 수행되며 마지막 비교표 셀이 실제 계산을 호출한다.
- Rerank는 주로 상위 정밀도와 순서를 개선하므로 P@5, MRR와 MAP를 함께 본다.


In [ ]:
# 질의별 예측 ID와 qrels를 `compute_metrics()`에서 상위 k 기준으로 비교한다.
# 전체 질의의 네 지표 평균이 각 검색 방법의 평가 딕셔너리가 된다.
import numpy as np


# 1. qrels 문자열을 관련 문서 ID와 등급의 딕셔너리로 변환한다.
def parse_relevant(relevant_str):
    # 입력 예시 `D6=3;D14=2`를 문서 ID와 관련성 등급의 딕셔너리로 변환한다.
    relevant_dict = {}
    for pair in relevant_str.split(";"):
        doc_id, grade_text = pair.split("=")
        grade = int(grade_text)
        # 0등급 문서가 포함되더라도 정답 hit로 세지 않고 1 이상만 보존한다.
        if grade > 0:
            relevant_dict[doc_id] = grade
    return relevant_dict


# 2. 예측 ID 순서와 정답 딕셔너리에서 P@k, R@k, RR와 AP@k를 계산한다.
def compute_metrics(predicted, relevant_dict, k=5):
    # predicted[:k]가 평가 대상이며 관련성 등급이 1 이상인 문서를 정답으로 처리한다.
    top_k = predicted[:k]
    hits = sum(doc_id in relevant_dict for doc_id in top_k)
    precision = hits / k

    total_relevant = len(relevant_dict)
    recall = hits / total_relevant if total_relevant else 0.0

    # RR은 첫 관련 문서의 순위 역수이므로 첫 정답이 1위이면 1.0이다.
    rr = next(
        (1 / rank for rank, doc_id in enumerate(predicted, start=1) if doc_id in relevant_dict),
        0.0,
    )

    # AP@k는 관련 문서를 만난 각 순위의 Precision을 min(관련 문서 수, k)로 나눈다.
    precision_sum = 0.0
    relevant_seen = 0
    for rank, doc_id in enumerate(top_k, start=1):
        if doc_id in relevant_dict:
            relevant_seen += 1
            precision_sum += relevant_seen / rank
    ap_denominator = min(total_relevant, k)
    ap = precision_sum / ap_denominator if ap_denominator else 0.0
    return precision, recall, rr, ap


# 3. 모든 질의의 네 지표를 평균내 검색 방식별 비교 딕셔너리를 반환한다.
def evaluate_all(method_results, queries_df, k=5):
    # 각 질의의 네 지표를 누적한 뒤 평균을 반환해 검색기 간 비교표에 사용한다.
    per_query_metrics = []
    for _, row in queries_df.iterrows():
        relevant_dict = parse_relevant(row["relevant_doc_ids"])
        predicted = method_results[row["query_id"]]
        per_query_metrics.append(compute_metrics(predicted, relevant_dict, k))

    metric_array = np.asarray(per_query_metrics, dtype=float)
    return {
        "P@k": metric_array[:, 0].mean(),
        "R@k": metric_array[:, 1].mean(),
        "MRR": metric_array[:, 2].mean(),
        "MAP": metric_array[:, 3].mean(),
    }


## 후보 생성기와 Rerank 성능 비교

BM25·Dense의 상위 5개와 Cohere 재정렬 상위 5개를 같은 표에 배치한다. Rerank는 1차 후보 집합 자체를 늘리지 않으므로 후보 집합 Recall의 상한은 바뀌지 않는다. 다만 후보 내부 순서가 달라지면 상위 5개만 보는 P@5, R@5, MRR와 MAP는 달라질 수 있다.

지표 개선은 Cohere 호출 비용, 후보 수에 따른 지연과 요청 한도를 함께 고려해 판단한다.

다음 번호의 문서 압축은 Rerank 결과를 이어받지 않는 별도 문서 표현 심화 분기이다. Rerank가 후보 순서를 바꾸는 반면 문서 압축은 인덱싱할 문서의 표현을 바꾼다.

### 코드 해석 순서

1. 두 1차 후보 목록도 상위 5개로 잘라 Rerank 결과와 반환 크기를 맞춘다.
2. 동일 qrels와 AP@5 정의로 세 방법의 평균 지표를 계산한다.
3. 같은 행에서 상위 순위 개선과 비용·지연 증가를 함께 비교한다.

### 결과 해석

- 외부 Rerank 경로를 실행하면 설정한 모델의 P@5, R@5, MRR와 MAP 표가 표시된다.
- 지표 개선이 작으면 후보 수, 문서 길이, 설정한 모델과 API 비용을 함께 재검토해야 한다.


In [ ]:
# 1. 두 1차 후보 목록도 상위 5개로 잘라 Rerank 결과와 반환 크기를 맞춘다.
bm25_results = {qid: lst[:5] for qid, lst in bm25_candidates.items()}
dense_results = {qid: lst[:5] for qid, lst in dense_candidates.items()}

# 2. 동일 qrels와 AP@5 정의로 세 방법의 평균 지표를 계산한다.
bm25_metrics = evaluate_all(bm25_results, queries_df)
dense_metrics = evaluate_all(dense_results, queries_df)
rerank_metrics = evaluate_all(rerank_results, queries_df)

# 3. 같은 행에서 상위 순위 개선과 비용·지연 증가를 함께 비교한다.
metrics_df = pd.DataFrame({
    'Metric': ['P@5', 'R@5', 'MRR', 'MAP'],
    'BM25': [bm25_metrics['P@k'], bm25_metrics['R@k'], bm25_metrics['MRR'], bm25_metrics['MAP']],
    'Dense': [dense_metrics['P@k'], dense_metrics['R@k'], dense_metrics['MRR'], dense_metrics['MAP']],
    'Rerank': [rerank_metrics['P@k'], rerank_metrics['R@k'], rerank_metrics['MRR'], rerank_metrics['MAP']]
})
metrics_df
